In [2]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 73 (delta 6), reused 12 (delta 2), pack-reused 56 (from 1)
Receiving objects: 100% (73/73), 43.00 MiB | 16.93 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Updating files: 100% (36/36), done.


In [7]:
%cd /content/Recommendation-Systems-benchmarking

/content/Recommendation-Systems-benchmarking


# Load datasets and candidate set

In [8]:
import pandas as pd
import numpy as np
import pickle

# Load train and test data
train= pd.read_csv("data/processed/train.csv")
test= pd.read_csv("data/processed/test.csv")

# Load movie metadata
movies=pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")

# Load fixed candidate set
candidate_sets=pd.read_pickle("data/processed/candidate_sets_100.pkl")

print("Data loaded successfully")

Data loaded successfully


# **Scoring functions**

#Popularity Model

In [10]:
# Count number of times movie appears in the train data.
movie_popularity=train.groupby("MovieID").size().to_dict()


def popularity_score(user_id, items):
    scores = []                       # Store popularity scores
    for movie_id in items:
        scores.append(movie_popularity.get(movie_id, 0))            # Get movie popluarity, if movie not found- assign 0
    return scores

In [45]:
# Example
items=[48,99,12,4000]
scores=popularity_score(user_id=1,items=items)

print("Candidate Movies:",items)
print("Popularity Scores:",scores)

Candidate Movies: [48, 99, 12, 4000]
Popularity Scores: [379, 50, 158, 0]


# CBF

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Replace missing genres
movies["Genres"] = movies["Genres"].fillna("")

# TF-IDF on genres
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["Genres"])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Map MovieID to dataframe index
movie_index = pd.Series(movies.index,index=movies["MovieID"])



In [54]:
def build_user_profile(user_id,train):
    user_movies = train[train["UserID"] == user_id]["MovieID"]        # Get all movies watched by user

    if user_movies.empty:           # if user has no movies
        return None

    idx = []                          # store indices of the watched movies
    for m in user_movies:
        if m in movie_index:
            idx.append(movie_index[m])

    if len(idx) == 0:           # if no movie indices return none
        return None

    return cosine_sim[idx].mean(axis=0)

In [63]:
# Returns similarity score for each candidate movie

def cbf_score(user_id, items):

    # Build user preference profile
    user_profile=build_user_profile(user_id, train)

    # If no user history
    if user_profile is None:
        return [0]*len(items)

    scores=[]

    # Get similarity score for each candidate movie
    for movie_id in items:
        if movie_id in movie_index:
            # Get similarity between user profile and movie
            score=user_profile[movie_index[movie_id]]
            scores.append(score)
        else:
            scores.append(0)
    return scores

In [64]:
# Example
items=[48,99,12,4000]
scores = cbf_score(user_id=1,items=items)
for movie_id, score in zip(items, scores):
    print(f"Movie ID: {movie_id} | CBF Score: {score:.4f}")

Movie ID: 48 | CBF Score: 0.3472
Movie ID: 99 | CBF Score: 0.0000
Movie ID: 12 | CBF Score: 0.0740
Movie ID: 4000 | CBF Score: 0.0000


# SVD

In [73]:
!pip install scikit-surprise
from surprise import Reader, Dataset, SVD

reader = Reader(rating_scale=(1, 5))

# Convert pandas dataframe to Surprise format
data = Dataset.load_from_df(train[["UserID", "MovieID", "Rating"]],reader)

# Build training set
trainset = data.build_full_trainset()

# Train SVD model
model = SVD(random_state=42)

model.fit(trainset)

print("SVD model trained successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.2 MB/s eta 0:00:00
SVD model trained successfully.


In [77]:
# SVD scoring function
def svd_score(user_id, items):
    scores = []
    # Predict rating for each candidate movie
    for movie_id in items:
        prediction=model.predict(user_id,movie_id)
        # Extract estimated rating
        score=prediction.est

        scores.append(score)

    return scores

In [78]:
# Example

items=[48,99,12,4000]

scores = svd_score(user_id=1,items=items)
for movie_id, score in zip(items, scores):
    print(f"Movie ID: {movie_id} | Predicted Rating: {score:.4f}")

Movie ID: 48 | Predicted Rating: 3.2342
Movie ID: 99 | Predicted Rating: 3.6977
Movie ID: 12 | Predicted Rating: 2.2905
Movie ID: 4000 | Predicted Rating: 3.6440


# Metrics

In [82]:

# Hit@10=Checks whether the true test item appears in the top 10 recommended items.

def hit_at_k(ranked_items, test_item, k=10):
    if test_item in ranked_items[:k]:
        return 1.0

    return 0.0

# NDCG@10 = Measures ranking quality. Higher score is given when the true item appears higher in the top 10.

def ndcg_at_k(ranked_items, test_item, k=10):
    top_k=ranked_items[:k]

    # Check whether the true item is recommended
    if test_item in top_k:

        # Get position of true item (starting from 1)
        rank=top_k.index(test_item)+1

        # Discount score based on ranking position
        return 1.0/np.log2(rank+1)

    return 0.0


In [83]:
# Evaluate a recommendation model.
    # This function will be used for all recommendation models.
    # It calculates the average Hit@10 and NDCG@10 using the fixed candidate set
def evaluate_model(score_fn, candidate_sets, k=10):
    hits=[]
    ndcgs=[]

    # Evaluate every user
    for user_id,candidate in candidate_sets.items():

        # Get 100 candidate movies
        items= candidate["items"]

        # Actual movie from test set
        test_item= candidate["test_item"]


        # Get model prediction scores
        scores= score_fn(user_id, items)


        # Rank movies according to predicted scores
        ranked_items= [item for item, score in sorted(
                zip(items, scores),
                key=lambda x: x[1],
                reverse=True)]


        # Calculate metrics
        hits.append(hit_at_k(ranked_items, test_item, k))

        ndcgs.append(
            ndcg_at_k(ranked_items, test_item, k)
        )
    # Return average performance over all users
    return {
        "Hit@10": np.mean(hits),
        "NDCG@10": np.mean(ndcgs)
    }

# Evaluate all 3 models

In [88]:
# Evaluate Popularity model

pop_results = evaluate_model(popularity_score,candidate_sets,k=10)

print("Popularity Baseline Results")
print(f"Hit@10 : {pop_results['Hit@10']:.4f}")
print(f"NDCG@10: {pop_results['NDCG@10']:.4f}")



Popularity Baseline Results
Hit@10 : 0.4733
NDCG@10: 0.2616


In [91]:
# CBF
cbf_results = evaluate_model(cbf_score,candidate_sets,k=10)
print("CBF Results")
print(f"Hit@10 : {cbf_results['Hit@10']:.4f}")
print(f"NDCG@10: {cbf_results['NDCG@10']:.4f}")

CBF Results
Hit@10 : 0.2459
NDCG@10: 0.1236


In [90]:
# SVD
svd_results=evaluate_model(svd_score,candidate_sets,k=10)

print("SVD Results")
print(f"Hit@10 : {svd_results['Hit@10']:.4f}")
print(f"NDCG@10: {svd_results['NDCG@10']:.4f}")

SVD Results
Hit@10 : 0.2685
NDCG@10: 0.1417


In [93]:
results = {
    "Popularity": pop_results,
    "CBF": cbf_results,
    "SVD": svd_results}
print("Traditional Model Evaluation Results")
for model_name, result in results.items():
    print(f"\n{model_name}")
    print(f"Hit@10 : {result['Hit@10']:.4f}")
    print(f"NDCG@10: {result['NDCG@10']:.4f}")

Traditional Model Evaluation Results

Popularity
Hit@10 : 0.4733
NDCG@10: 0.2616

CBF
Hit@10 : 0.2459
NDCG@10: 0.1236

SVD
Hit@10 : 0.2685
NDCG@10: 0.1417


# Multiple Runs

In [98]:
# Number of negative samples of each user (99 negative + 1 true test = 100 candidates)
N_NEGATIVES= 99

# Get all available movie IDs
all_movie_ids= movies["MovieID"].unique()

# Each user gets 1 positive item (item from test set), 99 negative items (Movies the user has not interacted with )
def build_candidate_sets(test_df, train_df, all_movie_ids, n_negatives=N_NEGATIVES, seed=42):
    rng = np.random.default_rng(seed)       # reproducible sampling
    candidates = {}

    # Store movies each user has already interacted with in training data
    user_seen = train_df.groupby("UserID")["MovieID"].apply(set)

    # Create candidate set for each test user
    for _, row in test_df.iterrows():

        # Get user ID and actual test movie (positive item)
        user_id = row["UserID"]
        test_item = row["MovieID"]

        # Include test item to prevent sampling it as a negative
        seen= user_seen.get(user_id, set()) | {test_item}

        # Select movies that the user has never interacted with
        neg_pool= np.setdiff1d(all_movie_ids, list(seen))

        # Randomly select negative samples
        n= min(n_negatives, len(neg_pool))
        negatives= rng.choice(neg_pool, size=n, replace=False)

        # Combine negative items with the true test item
        items = list(negatives) + [test_item]
        rng.shuffle(items)  # avoid test item always in same position
        candidates[user_id]= {"test_item": test_item, "items": items}

    return candidates

# Generate final candidate sets
candidate_sets = build_candidate_sets(test, train, all_movie_ids)
print(f"Built candidate sets for {len(candidate_sets)} users "
      f"({N_NEGATIVES} negatives + 1 test item each).")

Built candidate sets for 6040 users (99 negatives + 1 test item each).


In [99]:
results=[]

# Run evaluation 5 times with different candidate sets
for run in range(1, 6):
    print(f"Running experiment {run}/5")

    # Create new candidate sets with different seed
    candidate_sets=build_candidate_sets(test,train,all_movie_ids,n_negatives=99,seed=run)

    # Evaluate Popularity
    pop_results=evaluate_model(popularity_score,candidate_sets,k=10)

    # Evaluate CBF
    cbf_results = evaluate_model(cbf_score,candidate_sets,k=10)

    # Evaluate SVD
    svd_results=evaluate_model(svd_score,candidate_sets,k=10)


    # Save results
    results.append({
        "Run": run,

        "Popularity_Hit@10": pop_results["Hit@10"],
        "Popularity_NDCG@10": pop_results["NDCG@10"],

        "CBF_Hit@10": cbf_results["Hit@10"],
        "CBF_NDCG@10": cbf_results["NDCG@10"],

        "SVD_Hit@10": svd_results["Hit@10"],
        "SVD_NDCG@10": svd_results["NDCG@10"] })

# Convert to dataframe
results_df = pd.DataFrame(results)
results_df

Running experiment 1/5
Running experiment 2/5
Running experiment 3/5
Running experiment 4/5
Running experiment 5/5


,Run,Popularity_Hit@10,Popularity_NDCG@10,CBF_Hit@10,CBF_NDCG@10,SVD_Hit@10,SVD_NDCG@10
0,1,0.467053,0.258112,0.245695,0.125062,0.265232,0.139402
1,2,0.470861,0.257787,0.247682,0.124840,0.264735,0.138583
2,3,0.473344,0.260246,0.249669,0.126555,0.267715,0.140483
3,4,0.469702,0.258101,0.244371,0.125498,0.265563,0.140022
4,5,0.472682,0.260751,0.244536,0.122627,0.268874,0.141664


In [100]:
summary = results_df.drop(columns=["Run"]).agg(["mean","std"])
summary

,Popularity_Hit@10,Popularity_NDCG@10,CBF_Hit@10,CBF_NDCG@10,SVD_Hit@10,SVD_NDCG@10
mean,0.470728,0.258999,0.246391,0.124916,0.266424,0.140031
std,0.002512,0.001386,0.002259,0.001440,0.001781,0.001158


In [101]:
summary.to_csv("traditional_models_summary_results.csv")